#Kaggle Setup and Dataset download

In [1]:
!pip install kaggle -q

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sarahzs11","key":"2dc640254ee9b80334aa0bfbf784a0b6"}'}

In [4]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [5]:
import kagglehub

path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.
Path to dataset files: /kaggle/input/network-intrusion-dataset


In [6]:
import os

files = os.listdir(path)
print(files)

['Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']


#Dataset Cleaning

In [7]:
import pandas as pd
import os

all_files = os.listdir(path)

df_list = []

for file in all_files:
    if file.endswith(".csv"):
        file_path = os.path.join(path, file)
        print("Loading:", file)
        temp_df = pd.read_csv(file_path)
        df_list.append(temp_df)

# Combine everything
df = pd.concat(df_list, ignore_index=True)

print("Final shape:", df.shape)
df.head()

Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv
Final shape: (2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,22,160,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
# Reduce memory usage by converting float64 → float32
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')

# Convert int64 → int32
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = df[col].astype('int32')

print("Memory optimized")

Memory optimized


In [10]:
import numpy as np

# Fix column names
df.columns = df.columns.str.strip()

# Replace infinities
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop NaN rows
df.dropna(inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

print("Cleaned shape:", df.shape)

Cleaned shape: (2498185, 79)


In [11]:
print(df['Label'].value_counts())

Label
BENIGN                        2072444
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [12]:
# Convert to binary classification
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print(df['Label'].value_counts())

Label
0    2072444
1     425741
Name: count, dtype: int64


In [13]:
from sklearn.utils import resample

# Separate classes
df_majority = df[df['Label'] == 0]
df_minority = df[df['Label'] == 1]

# Downsample majority class
df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

# Combine
df_balanced = pd.concat([df_majority_downsampled, df_minority])

print(df_balanced['Label'].value_counts())

Label
0    425741
1    425741
Name: count, dtype: int64


#Test train split

In [14]:
from sklearn.model_selection import train_test_split

X = df_balanced.drop(columns=['Label'])
y = df_balanced['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (681185, 78)
Test shape: (170297, 78)


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling complete")

Scaling complete


#IDS detection models

##Train Logistic Regression Model

In [16]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, n_jobs=-1)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

###Evaluation

In [17]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_log))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("ROC-AUC:", roc_auc_score(y_test, y_pred_log))

Logistic Regression Results
              precision    recall  f1-score   support

           0       0.97      0.93      0.95     85149
           1       0.93      0.98      0.95     85148

    accuracy                           0.95    170297
   macro avg       0.95      0.95      0.95    170297
weighted avg       0.95      0.95      0.95    170297

Confusion Matrix:
[[79169  5980]
 [ 2078 83070]]
ROC-AUC: 0.9526828007107179


##Random forest Model

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

###Evaluation

In [19]:
print("Random Forest Results")
print(classification_report(y_test, y_pred_rf))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("ROC-AUC:", roc_auc_score(y_test, y_pred_rf))

Random Forest Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85149
           1       1.00      1.00      1.00     85148

    accuracy                           1.00    170297
   macro avg       1.00      1.00      1.00    170297
weighted avg       1.00      1.00      1.00    170297

Confusion Matrix:
[[85048   101]
 [   98 85050]]
ROC-AUC: 0.99883145338796


#Attack Type Classification using random forest

In [8]:
import pandas as pd
import os

df_list = []

for file in os.listdir(path):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(path, file))
        df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print(df['Label'].value_counts())

Shape: (2830743, 79)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [9]:
import numpy as np

# Convert float64 → float32
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')

# Convert int64 → int32
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = df[col].astype('int32')

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("Memory optimized")

Memory optimized


In [11]:
# Multiclass target
y_multi = df['Label']
X_multi = df.drop(columns=['Label'])  # if BinaryLabel exists

In [12]:
from sklearn.model_selection import train_test_split

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi,
    y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_multi = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_multi.fit(X_train_m, y_train_m)

y_pred_multi = rf_multi.predict(X_test_m)

In [15]:
from sklearn.metrics import classification_report

print("Multiclass Results")
print(classification_report(y_test_m, y_pred_multi))

Multiclass Results
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    454265
                       Bot       0.86      0.76      0.81       391
                      DDoS       1.00      1.00      1.00     25605
             DoS GoldenEye       1.00      0.99      1.00      2059
                  DoS Hulk       1.00      1.00      1.00     46025
          DoS Slowhttptest       0.99      0.99      0.99      1100
             DoS slowloris       1.00      1.00      1.00      1159
               FTP-Patator       1.00      1.00      1.00      1587
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.57      0.73         7
                  PortScan       0.99      1.00      1.00     31761
               SSH-Patator       1.00      1.00      1.00      1180
  Web Attack � Brute Force       0.73      0.81      0.77       301
Web Attack � Sql Injection  

In [ ]:
# Run this in a Colab cell before downloading
!pip install nbformat

!ls "/content/drive/MyDrive/Colab Notebooks/"

import nbformat
import json

# Read your notebook
with open("/content/drive/MyDrive/Colab Notebooks/Project_IDS_ML.ipynb", 'r') as f:
    nb = nbformat.read(f, as_version=4)

# Remove widget metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# Save the cleaned notebook
with open('/content/Project_IDS_ML.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Notebook cleaned! Download 'your_notebook_cleaned.ipynb'")